Name: SIDDHARTH BALAJI

BITS ID: 2026AJ05658

MACHINE LEARNING - Assignment 2

Loading the data, removing duplicates and checking the basic details of the dataset

In [7]:
import re
import zipfile
import numpy as np
import pandas as pd

with zipfile.ZipFile("spambase.zip") as z:
    z.extractall("spambase_raw")

symbol_map = {";": "semicolon", "(": "paren", "[": "bracket", "!": "exclam", "$": "dollar", "#": "hash"}

feature_names = []
with open("spambase_raw/spambase.names") as f:
    for line in f:
        match = re.match(r"^(.+?):\s+continuous", line)
        if match:
            col = match.group(1)
            for sym, word in symbol_map.items():
                col = col.replace(sym, word)
            feature_names.append(col)

target_col = "is_spam"
raw_df = pd.read_csv("spambase_raw/spambase.data", header=None, names=feature_names + [target_col])

print("Raw shape:", raw_df.shape)
print("Missing values:", raw_df.isna().sum().sum())
print("Duplicate rows:", raw_df.duplicated().sum())

emails_df = raw_df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", emails_df.shape)
print("Number of features:", emails_df.shape[1] - 1)
print("\nClass distribution:")
print(emails_df[target_col].value_counts())
print(emails_df[target_col].value_counts(normalize=True).round(3))

emails_df.head()

Raw shape: (4601, 58)
Missing values: 0
Duplicate rows: 391
Shape after removing duplicates: (4210, 58)
Number of features: 57

Class distribution:
is_spam
0    2531
1    1679
Name: count, dtype: int64
is_spam
0    0.601
1    0.399
Name: proportion, dtype: float64


,word_freq_make,word_freq_address,word_freq_all,word_freq_3d,word_freq_our,word_freq_over,word_freq_remove,word_freq_internet,word_freq_order,word_freq_mail,...,char_freq_semicolon,char_freq_paren,char_freq_bracket,char_freq_exclam,char_freq_dollar,char_freq_hash,capital_run_length_average,capital_run_length_longest,capital_run_length_total,is_spam
0,0.00,0.64,0.64,0.0,0.32,0.00,0.00,0.00,0.00,0.00,...,0.00,0.000,0.0,0.778,0.000,0.000,3.756,61,278,1
1,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,...,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028,1
2,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,...,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259,1
3,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.137,0.0,0.137,0.000,0.000,3.537,40,191,1
4,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.135,0.0,0.135,0.000,0.000,3.537,40,191,1


Splitting the data and creating test_data.csv:

In [8]:
import os
import joblib
import sklearn
from sklearn.model_selection import train_test_split

feature_matrix = emails_df.drop(columns=[target_col])
label_vector = emails_df[target_col]

# Stratified 80/20 split with a fixed seed
X_train, X_test, y_train, y_test = train_test_split(
    feature_matrix,
    label_vector,
    test_size=0.2,
    stratify=label_vector,
    random_state=42,
)

# Saving the held-out test set (features + target) for the Streamlit app
test_data = X_test.copy()
test_data[target_col] = y_test
test_data.to_csv("test_data.csv", index=False)

print("Train rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])
print("Test spam share:", round(y_test.mean(), 3))

Train rows: 3368
Test rows: 842
Test spam share: 0.399


Training, Evaluating and Saving all the models:

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
)

# One pipeline per classifier, with scaling only wherever required
model_zoo = {
    "Logistic Regression": Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    "Decision Tree": Pipeline([
        ("clf", DecisionTreeClassifier(random_state=42)),
    ]),
    "KNN": Pipeline([
        ("scale", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=5)),
    ]),
    "Naive Bayes": Pipeline([
        ("clf", GaussianNB()),
    ]),
    "Support Vector Machine": Pipeline([
        ("scale", StandardScaler()),
        ("clf", SVC(probability=True, random_state=42)),
    ]),
}

os.makedirs("model", exist_ok=True)
metric_rows = []

for model_name, pipe in model_zoo.items():
    # Fitting on training data only
    pipe.fit(X_train, y_train)

    # Predictions and spam-class probabilities on the test set
    predicted_labels = pipe.predict(X_test)
    spam_probability = pipe.predict_proba(X_test)[:, 1]

    metric_rows.append({
        "ML Model Name": model_name,
        "Accuracy": accuracy_score(y_test, predicted_labels),
        "AUC": roc_auc_score(y_test, spam_probability),
        "Precision": precision_score(y_test, predicted_labels),
        "Recall": recall_score(y_test, predicted_labels),
        "F1": f1_score(y_test, predicted_labels),
        "MCC": matthews_corrcoef(y_test, predicted_labels),
    })

    # Saving the fitted pipeline to the model folder
    file_stub = model_name.lower().replace(" ", "_")
    joblib.dump(pipe, f"model/{file_stub}.joblib")

comparison_table = pd.DataFrame(metric_rows).set_index("ML Model Name").round(4)
comparison_table.to_csv("model/metrics_summary.csv")
comparison_table

,Accuracy,AUC,Precision,Recall,F1,MCC
ML Model Name,,,,,,
Logistic Regression,0.9347,0.9768,0.9297,0.9048,0.9170,0.8634
Decision Tree,0.9074,0.9053,0.8817,0.8869,0.8843,0.8071
KNN,0.9133,0.9546,0.9148,0.8631,0.8882,0.8184
Naive Bayes,0.8361,0.9525,0.7240,0.9524,0.8226,0.6975
Support Vector Machine,0.9323,0.9772,0.9319,0.8958,0.9135,0.8584


Packaging the files and recording library versions

In [10]:
import shutil
import pandas as pd
import numpy as np
from google.colab import files

print("scikit-learn:", sklearn.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("joblib:", joblib.__version__)

# Bundling the model folder and downloading it with the test file for ease of use
shutil.make_archive("model_files", "zip", "model")
files.download("model_files.zip")
files.download("test_data.csv")

scikit-learn: 1.6.1
pandas: 2.2.3
numpy: 2.4.6
joblib: 1.6.0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>